In [0]:
# Service Principal credentials
application_id = "************************"
authentication_key = "*******************"
tenant_id = "**************************"

# Set Spark config for ADLS access
spark.conf.set("fs.azure.account.auth.type.petroflowstorage.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.petroflowstorage.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.petroflowstorage.dfs.core.windows.net", application_id)
spark.conf.set("fs.azure.account.oauth2.client.secret.petroflowstorage.dfs.core.windows.net", authentication_key)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.petroflowstorage.dfs.core.windows.net", "https://login.microsoftonline.com/" + tenant_id + "/oauth2/token")

print("✅ Spark config set!")

✅ Spark config set!


In [0]:
bronze_path = "abfss://bronze@petroflowstorage.dfs.core.windows.net/oil-prices/oil_prices_raw.json"

from pyspark.sql.functions import explode, col

# Read raw JSON
df_raw = spark.read \
    .option("multiline", "true") \
    .json(bronze_path)

# Explode nested response.data array
df_exploded = df_raw \
    .select(explode(col("response.data")).alias("record"))

# Flatten the struct
df_flat = df_exploded.select(
    col("record.period").alias("period"),
    col("record.value").alias("value"),
    col("record.series").alias("series"),
    col("record.series-description").alias("series_description"),
    col("record.units").alias("units"),
    col("record.duoarea").alias("area"),
    col("record.product-name").alias("product_name")
)

df_flat.printSchema()
df_flat.show(10)
print(f"Total records after explode: {df_flat.count()}")

root
 |-- period: string (nullable = true)
 |-- value: string (nullable = true)
 |-- series: string (nullable = true)
 |-- series_description: string (nullable = true)
 |-- units: string (nullable = true)
 |-- area: string (nullable = true)
 |-- product_name: string (nullable = true)

+----------+-----+--------------------+--------------------+-----+-----+------------+
|    period|value|              series|  series_description|units| area|product_name|
+----------+-----+--------------------+--------------------+-----+-----+------------+
|2025-11-04|2.538|EER_EPD2DC_PF4_Y0...|Los Angeles, CA U...|$/GAL|Y05LA| Carb Diesel|
|2025-11-05| 2.63|EER_EPD2DC_PF4_Y0...|Los Angeles, CA U...|$/GAL|Y05LA| Carb Diesel|
|2025-11-06|2.701|EER_EPD2DC_PF4_Y0...|Los Angeles, CA U...|$/GAL|Y05LA| Carb Diesel|
|2025-11-07|2.683|EER_EPD2DC_PF4_Y0...|Los Angeles, CA U...|$/GAL|Y05LA| Carb Diesel|
|2025-11-10|  2.7|EER_EPD2DC_PF4_Y0...|Los Angeles, CA U...|$/GAL|Y05LA| Carb Diesel|
|2025-11-12|2.674|EER_EPD2

In [0]:
from pyspark.sql.functions import *

df_clean = df_flat \
    .filter(col("value").isNotNull()) \
    .filter(col("period").isNotNull()) \
    .filter(col("value").cast("double") > 0) \
    .withColumn("price_usd", round(col("value").cast("double"), 2)) \
    .withColumn("trade_date", to_date(col("period"), "yyyy-MM-dd")) \
    .withColumn("energy_type", lit("CRUDE_OIL")) \
    .withColumn("source_system", lit("EIA_API")) \
    .withColumn("ingestion_date", current_date()) \
    .select(
        "trade_date",
        "price_usd",
        "series",
        "series_description",
        "units",
        "area",
        "product_name",
        "energy_type",
        "source_system",
        "ingestion_date"
    )

df_clean.show(10)
print(f"Clean records: {df_clean.count()}")

+----------+---------+--------------------+--------------------+-----+-----+------------+-----------+-------------+--------------+
|trade_date|price_usd|              series|  series_description|units| area|product_name|energy_type|source_system|ingestion_date|
+----------+---------+--------------------+--------------------+-----+-----+------------+-----------+-------------+--------------+
|2025-11-04|     2.54|EER_EPD2DC_PF4_Y0...|Los Angeles, CA U...|$/GAL|Y05LA| Carb Diesel|  CRUDE_OIL|      EIA_API|    2026-06-04|
|2025-11-05|     2.63|EER_EPD2DC_PF4_Y0...|Los Angeles, CA U...|$/GAL|Y05LA| Carb Diesel|  CRUDE_OIL|      EIA_API|    2026-06-04|
|2025-11-06|      2.7|EER_EPD2DC_PF4_Y0...|Los Angeles, CA U...|$/GAL|Y05LA| Carb Diesel|  CRUDE_OIL|      EIA_API|    2026-06-04|
|2025-11-07|     2.68|EER_EPD2DC_PF4_Y0...|Los Angeles, CA U...|$/GAL|Y05LA| Carb Diesel|  CRUDE_OIL|      EIA_API|    2026-06-04|
|2025-11-10|      2.7|EER_EPD2DC_PF4_Y0...|Los Angeles, CA U...|$/GAL|Y05LA| Carb D

In [0]:
silver_path = "abfss://silver@petroflowstorage.dfs.core.windows.net/oil-prices/"

df_clean.write \
    .mode("overwrite") \
    .parquet(silver_path)

print("Oil prices written to silver! ✅")

Oil prices written to silver! ✅


In [0]:
df_verify = spark.read.parquet(silver_path)
df_verify.show(10)
print(f"Total records in silver: {df_verify.count()}")

+----------+---------+--------------------+--------------------+-----+-----+------------+-----------+-------------+--------------+
|trade_date|price_usd|              series|  series_description|units| area|product_name|energy_type|source_system|ingestion_date|
+----------+---------+--------------------+--------------------+-----+-----+------------+-----------+-------------+--------------+
|2025-11-04|     2.54|EER_EPD2DC_PF4_Y0...|Los Angeles, CA U...|$/GAL|Y05LA| Carb Diesel|  CRUDE_OIL|      EIA_API|    2026-06-04|
|2025-11-05|     2.63|EER_EPD2DC_PF4_Y0...|Los Angeles, CA U...|$/GAL|Y05LA| Carb Diesel|  CRUDE_OIL|      EIA_API|    2026-06-04|
|2025-11-06|      2.7|EER_EPD2DC_PF4_Y0...|Los Angeles, CA U...|$/GAL|Y05LA| Carb Diesel|  CRUDE_OIL|      EIA_API|    2026-06-04|
|2025-11-07|     2.68|EER_EPD2DC_PF4_Y0...|Los Angeles, CA U...|$/GAL|Y05LA| Carb Diesel|  CRUDE_OIL|      EIA_API|    2026-06-04|
|2025-11-10|      2.7|EER_EPD2DC_PF4_Y0...|Los Angeles, CA U...|$/GAL|Y05LA| Carb D